# RAGAS 深度評測：金融 RAG 共用基準

本 Notebook 與其他兩套工具共用同一份金融文件、同一組問題集與同一個 baseline RAG。

## 評測流程圖

```mermaid
flowchart LR
    A["載入金融 PDF 與 Benchmark 題庫"] --> B["建立共用 RAG Baseline"]
    B --> C["產生每題檢索上下文與回答"]
    C --> D["執行工具評測"]
    D --> E["輸出 CSV 供橫向比較"]
```


### Cell 1 說明：安裝與版本固定

這一格會安裝 RAGAS 評測與向量檢索所需套件，並固定主要版本。

重點：
- 固定 `ragas==0.4.3`。
- 使用 OpenAI 官方 `openai` SDK 建立 embedding 向量。
- 補上 `python-dotenv`，確保 `.env` 讀取一致。


In [13]:
!uv add ragas==0.4.3 openai pypdf scikit-learn pandas numpy python-dotenv


Resolved 216 packages in 10ms
Audited 214 packages in 233ms


### Cell 2 說明：建立 Vector RAG baseline（text-embedding-3-large + Top-5）

這一格改為 **向量檢索 baseline**：

1. 載入 PDF 與 benchmark 題庫。
2. 文本切塊後，使用 `text-embedding-3-large` 產生 chunk embeddings。
3. 以 cosine similarity 做向量檢索，取 `top_k=5`。
4. 用檢索到的內容組合 baseline 回答，輸出 `rag_df`。

補充：
- 內建 embedding cache（JSON）以減少重跑成本。


In [14]:
from dotenv import load_dotenv
import hashlib
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
from openai import OpenAI
from pypdf import PdfReader

PROJECT_ROOT = Path(r"/Users/caocharles/Library/CloudStorage/OneDrive-個人/GitHub/claude_test/llm-paper-obsidian")
load_dotenv(PROJECT_ROOT / ".env")
load_dotenv(PROJECT_ROOT / "lpdd/.env")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Please set OPENAI_API_KEY before running vector RAG baseline.")

openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL") or os.getenv("OPENAI_API_BASE") or None,
)

PDF_PATH = PROJECT_ROOT / "docs/Benchmark-Governance/data/financial-stability-report-20211108.pdf"
BENCH_PATH = PROJECT_ROOT / "docs/Benchmark-Governance/data/finance-rag-benchmark.json"
RESULT_DIR = PROJECT_ROOT / "docs/Benchmark-Governance/data/results"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

EMBED_MODEL = "text-embedding-3-large"
EMBED_CACHE_PATH = RESULT_DIR / f"embedding_cache_{EMBED_MODEL}.json"

assert PDF_PATH.exists(), f"Missing PDF: {PDF_PATH}"
assert BENCH_PATH.exists(), f"Missing benchmark: {BENCH_PATH}"

reader = PdfReader(str(PDF_PATH))
raw_text = "\n".join((p.extract_text() or "") for p in reader.pages)
raw_text = " ".join(raw_text.split())

chunk_size = 1200
stride = 900
chunks = []
for i in range(0, max(len(raw_text) - chunk_size + 1, 1), stride):
    part = raw_text[i : i + chunk_size]
    if len(part) >= 300:
        chunks.append(part)

if not chunks:
    raise RuntimeError("No chunks generated from PDF. Please check PDF extraction.")

with open(BENCH_PATH, "r", encoding="utf-8") as f:
    benchmark = json.load(f)

if EMBED_CACHE_PATH.exists():
    with open(EMBED_CACHE_PATH, "r", encoding="utf-8") as f:
        embedding_cache = json.load(f)
else:
    embedding_cache = {}


def _text_key(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def _normalize_rows(vectors: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    return vectors / np.clip(norms, 1e-12, None)


def embed_texts(texts: list[str], batch_size: int = 32) -> np.ndarray:
    keys = [_text_key(t) for t in texts]
    missing = [(k, t) for k, t in zip(keys, texts) if k not in embedding_cache]

    for i in range(0, len(missing), batch_size):
        batch = missing[i : i + batch_size]
        batch_keys = [k for k, _ in batch]
        batch_texts = [t for _, t in batch]

        response = openai_client.embeddings.create(model=EMBED_MODEL, input=batch_texts)
        data_sorted = sorted(response.data, key=lambda x: x.index)
        for k, d in zip(batch_keys, data_sorted):
            embedding_cache[k] = d.embedding

    if missing:
        with open(EMBED_CACHE_PATH, "w", encoding="utf-8") as f:
            json.dump(embedding_cache, f)

    mat = np.asarray([embedding_cache[k] for k in keys], dtype=np.float32)
    return _normalize_rows(mat)


chunk_embeddings = embed_texts(chunks)


def retrieve(query: str, top_k: int = 5):
    query_vec = embed_texts([query])[0]
    scores = chunk_embeddings @ query_vec
    order = np.argsort(scores)[::-1][:top_k]
    return [chunks[i] for i in order], [float(scores[i]) for i in order]


def generate_baseline_answer(query: str, retrieved_chunks: list[str]) -> str:
    text = " ".join(retrieved_chunks[:2])
    sents = [s.strip() for s in text.replace("?", ".").split(".") if len(s.strip()) > 30]
    selected = sents[:3]
    if not selected:
        return "No grounded answer generated from retrieved context."
    return " ".join(selected)


rows = []
for item in benchmark:
    contexts, scores = retrieve(item["question"], top_k=5)
    answer = generate_baseline_answer(item["question"], contexts)
    rows.append(
        {
            "id": item["id"],
            "question": item["question"],
            "ground_truth": item["ground_truth"],
            "topic": item["topic"],
            "retrieved_contexts": contexts,
            "retrieval_scores": scores,
            "answer": answer,
        }
    )

rag_df = pd.DataFrame(rows)
print(f"chunks={len(chunks)}, embed_cache_entries={len(embedding_cache)}")
rag_df.head(3)


chunks=214, embed_cache_entries=222


,id,question,ground_truth,topic,retrieved_contexts,retrieval_scores,answer
0,FSR-Q01,Financial Stability Report 的主要目的為何？,該報告用於呈現聯準會對美國金融系統韌性的評估，並提升透明度與公眾理解。,purpose,[t financial hardship. Monitoring and assessin...,"[0.6142961978912354, 0.6136758923530579, 0.568...",Monitoring and assessing financial stability a...
1,FSR-Q02,報告如何描述金融穩定與聯準會雙重使命的關係？,金融穩定有助於實現充分就業與物價穩定；金融不穩定會干擾信貸流動並造成失業與經濟困難。,purpose,[lable on the Board’s website; see Board of Go...,"[0.5444421768188477, 0.5427120923995972, 0.528...",lable on the Board’s website; see Board of Gov...
2,FSR-Q03,報告框架如何區分 shocks 與 vulnerabilities？,shocks 是難以預測的突發事件；vulnerabilities 是隨時間累積、在壓力情境...,framework,[it provision and payment services. By contras...,"[0.5232314467430115, 0.4914538860321045, 0.491...","it provision and payment services By contrast,..."


### Cell 3 說明：執行 RAGAS 核心指標評測並輸出結果

這一格使用 **RAGAS 0.4.3 相容寫法**（修正新版 API 變動）：

重點：
- `llm_factory(...)` 必須傳入 `OpenAI client`。
- `answer_relevancy` 在本版仍需相容的 embedding wrapper，因此採用 legacy `embedding_factory("text-embedding-3-small")`。
- 指標採用：`faithfulness`、`answer_relevancy`、`context_precision`、`context_recall`、`context_entity_recall`。

評測完成後會輸出：
- `result_df`：逐題分數明細。
- `ragas_finance_results.csv`：供跨工具比較。


In [15]:
from copy import deepcopy
from openai import OpenAI

from ragas import evaluate
from ragas.dataset_schema import EvaluationDataset, SingleTurnSample
from ragas.llms import llm_factory
from ragas.embeddings.base import embedding_factory

# NOTE:
# For ragas==0.4.3, evaluate() expects classic Metric objects.
# Use metric objects from ragas.metrics (deprecated path) for compatibility.
from ragas.metrics import (
    faithfulness as m_faithfulness,
    answer_relevancy as m_answer_relevancy,
    context_precision as m_context_precision,
    context_recall as m_context_recall,
    context_entity_recall as m_context_entity_recall,
)

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Please set OPENAI_API_KEY before running RAGAS evaluation.")

openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL") or os.getenv("OPENAI_API_BASE") or None,
)

# Use a cheaper judge model for tutorial reproducibility.
judge_llm = llm_factory(model="gpt-4o-mini", provider="openai", client=openai_client)

# Compatibility wrapper for answer_relevancy in ragas 0.4.3.
judge_embeddings = embedding_factory("text-embedding-3-small")

faithfulness = deepcopy(m_faithfulness)
faithfulness.llm = judge_llm

answer_relevancy = deepcopy(m_answer_relevancy)
answer_relevancy.llm = judge_llm
answer_relevancy.embeddings = judge_embeddings

context_precision = deepcopy(m_context_precision)
context_precision.llm = judge_llm

context_recall = deepcopy(m_context_recall)
context_recall.llm = judge_llm

context_entity_recall = deepcopy(m_context_entity_recall)
context_entity_recall.llm = judge_llm

samples = []
for r in rows:
    samples.append(
        SingleTurnSample(
            user_input=r["question"],
            response=r["answer"],
            retrieved_contexts=r["retrieved_contexts"],
            reference=r["ground_truth"],
        )
    )

dataset = EvaluationDataset(samples=samples)
metrics = [
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
    context_entity_recall,
]

result = evaluate(dataset=dataset, metrics=metrics, show_progress=True)
result_df = result.to_pandas().reset_index(drop=True)

# Align key columns with our baseline dataframe (overwrite if already present).
result_df["id"] = rag_df["id"].reset_index(drop=True)
result_df["question"] = rag_df["question"].reset_index(drop=True)
result_df["reference"] = rag_df["ground_truth"].reset_index(drop=True)
result_df["output"] = rag_df["answer"].reset_index(drop=True)
result_df["tool"] = "ragas"

front_cols = ["tool", "id", "question", "reference", "output"]
other_cols = [c for c in result_df.columns if c not in front_cols]
result_df = result_df[front_cols + other_cols]

result_df.to_csv(RESULT_DIR / "ragas_finance_results.csv", index=False)
result_df.head()


/var/folders/pg/d9j7f7yn2nlbm579gfzrnvk80000gp/T/ipykernel_606/1836800306.py:12: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
/var/folders/pg/d9j7f7yn2nlbm579gfzrnvk80000gp/T/ipykernel_606/1836800306.py:12: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
/var/folders/pg/d9j7f7yn2nlbm579gfzrnvk80000gp/T/ipykernel_606/1836800306.py:12: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
/var

,tool,id,question,reference,output,user_input,retrieved_contexts,response,faithfulness,answer_relevancy,context_precision,context_recall,context_entity_recall
0,ragas,FSR-Q01,Financial Stability Report 的主要目的為何？,該報告用於呈現聯準會對美國金融系統韌性的評估，並提升透明度與公眾理解。,Monitoring and assessing financial stability a...,Financial Stability Report 的主要目的為何？,[t financial hardship. Monitoring and assessin...,Monitoring and assessing financial stability a...,1.00,0.553586,1.0,1.0,0.0
1,ragas,FSR-Q02,報告如何描述金融穩定與聯準會雙重使命的關係？,金融穩定有助於實現充分就業與物價穩定；金融不穩定會干擾信貸流動並造成失業與經濟困難。,lable on the Board’s website; see Board of Gov...,報告如何描述金融穩定與聯準會雙重使命的關係？,[lable on the Board’s website; see Board of Go...,lable on the Board’s website; see Board of Gov...,1.00,0.000000,1.0,1.0,0.0
2,ragas,FSR-Q03,報告框架如何區分 shocks 與 vulnerabilities？,shocks 是難以預測的突發事件；vulnerabilities 是隨時間累積、在壓力情境...,"it provision and payment services By contrast,...",報告框架如何區分 shocks 與 vulnerabilities？,[it provision and payment services. By contras...,"it provision and payment services By contrast,...",1.00,0.403357,1.0,1.0,0.4
3,ragas,FSR-Q04,報告中的第一類脆弱性是什麼？,第一類是 elevated valuation pressures，指資產估值相對基本面偏高...,"re not transparent to counterparties, and affe...",報告中的第一類脆弱性是什麼？,"[re not transparent to counterparties, and aff...","re not transparent to counterparties, and affe...",0.75,0.000000,0.0,0.0,0.0
4,ragas,FSR-Q05,第二類脆弱性在報告中如何定義？,第二類是企業與家戶過度借貸，當收入下滑或資產價值下降時容易造成支出縮減與違約風險。,translates to considering cyber risk to ﬁ nanc...,第二類脆弱性在報告中如何定義？,[translates to considering cyber risk to ﬁ nan...,translates to considering cyber risk to ﬁ nanc...,0.80,0.000000,0.0,0.0,0.0


In [25]:
result_df.query("id == 'FSR-Q01'")

,tool,id,question,reference,output,user_input,retrieved_contexts,response,faithfulness,answer_relevancy,context_precision,context_recall,context_entity_recall
0,ragas,FSR-Q01,Financial Stability Report 的主要目的為何？,該報告用於呈現聯準會對美國金融系統韌性的評估，並提升透明度與公眾理解。,Monitoring and assessing financial stability a...,Financial Stability Report 的主要目的為何？,[t financial hardship. Monitoring and assessin...,Monitoring and assessing financial stability a...,1.0,0.553586,1.0,1.0,0.0


In [29]:
result_df.query("id == 'FSR-Q01'").user_input.values[0], result_df.query("id == 'FSR-Q01'").reference.values[0], result_df.query("id == 'FSR-Q01'").output.values[0]

('Financial Stability Report 的主要目的為何？',
 '該報告用於呈現聯準會對美國金融系統韌性的評估，並提升透明度與公眾理解。',
 'Monitoring and assessing financial stability also support the Federal Reserve’s regulatory and supervisory activities, which promote the safety and soundness of our nation’s banks and other impor- tant financial institutions Information gathered while monitoring the stability of the finan- cial system helps the Federal Reserve develop its view of the salient risks to be included in the scenarios of the stress tests and its setting of the countercyclical capital buffer (CCyB) 1 The Board’s Financial Stability Report is similar to those published by other central banks and complements the annual report of the Financial Stability Oversight Council (FSOC), which is chaired by the Secretary of the Treasury and includes the Federal Reserve Board Chair and other financial regulators')

### Cell 4 說明：快速查看整體分數摘要

這一格先看全局平均，快速回答「目前 baseline 到底卡在哪」：

- `faithfulness` 低：答案常偏離證據。
- `answer_relevancy` 低：答案切題度不足。
- `context_precision` 低：檢索回來內容不夠精準。
- `context_recall` 低：檢索漏掉關鍵證據。
- `context_entity_recall` 低：答案涉及的關鍵實體沒有被充分覆蓋。


In [16]:
result_df.select_dtypes(include=["number"]).mean(numeric_only=True).sort_index()


answer_relevancy         0.198841
context_entity_recall    0.075000
context_precision        0.733333
context_recall           0.750000
faithfulness             0.943750
dtype: float64

### Cell 5 說明：逐題分數與風險排序

這一格把每題分數攤開，方便你找到「拖累平均分」的題目。

輸出：
- `metric_summary_df`：整體平均。
- `per_question_df`：逐題平均與排序。


In [17]:
preferred_metrics = [
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_recall",
    "context_entity_recall",
]
metric_cols = [c for c in preferred_metrics if c in result_df.columns]
if not metric_cols:
    metric_cols = result_df.select_dtypes(include=["number"]).columns.tolist()

metric_summary_df = result_df[metric_cols].mean(numeric_only=True).to_frame("mean_score").sort_index()

per_question_df = result_df[["id", "question"] + metric_cols].copy()
per_question_df["mean_score"] = per_question_df[metric_cols].mean(axis=1, skipna=True)
per_question_df = per_question_df.sort_values("mean_score", ascending=True).reset_index(drop=True)

display(metric_summary_df)
display(per_question_df)


,mean_score
answer_relevancy,0.198841
context_entity_recall,0.075000
context_precision,0.733333
context_recall,0.750000
faithfulness,0.943750


,id,question,faithfulness,answer_relevancy,context_precision,context_recall,context_entity_recall,mean_score
0,FSR-Q04,報告中的第一類脆弱性是什麼？,0.75,0.000000,0.000000,0.0,0.0,0.150000
1,FSR-Q05,第二類脆弱性在報告中如何定義？,0.80,0.000000,0.000000,0.0,0.0,0.160000
2,FSR-Q06,第三類脆弱性與金融機構行為有何關聯？,1.00,0.000000,0.866667,1.0,0.0,0.573333
3,FSR-Q02,報告如何描述金融穩定與聯準會雙重使命的關係？,1.00,0.000000,1.000000,1.0,0.0,0.600000
4,FSR-Q07,第四類 funding risk 為何會導致 run 風險？,1.00,0.000000,1.000000,1.0,0.0,0.600000
5,FSR-Q01,Financial Stability Report 的主要目的為何？,1.00,0.553586,1.000000,1.0,0.0,0.710717
6,FSR-Q03,報告框架如何區分 shocks 與 vulnerabilities？,1.00,0.403357,1.000000,1.0,0.4,0.760671
7,FSR-Q08,報告提到的 CCyB 與壓力測試有何關係？,1.00,0.633785,1.000000,1.0,0.2,0.766757


### Cell 6 說明：把分數轉成可行動診斷

這一格把分數映射為「可能根因」，讓你知道下一步應優先改哪一段：

- 檢索覆蓋不足（Recall 問題）
- 檢索精準不足（Precision 問題）
- 生成/對齊問題（Faithfulness/Relevancy 問題）


In [18]:
diagnostic_df = per_question_df.copy()

for c in ["faithfulness", "answer_relevancy", "context_precision", "context_recall", "context_entity_recall"]:
    if c not in diagnostic_df.columns:
        diagnostic_df[c] = np.nan

f = diagnostic_df["faithfulness"].fillna(1.0)
a = diagnostic_df["answer_relevancy"].fillna(1.0)
cp = diagnostic_df["context_precision"].fillna(1.0)
cr = diagnostic_df["context_recall"].fillna(1.0)

conditions = [
    (cr < 0.5) & (cp >= 0.5),
    (cp < 0.5) & (cr >= 0.5),
    (f < 0.5) | (a < 0.5),
]
choices = [
    "retrieval_recall_issue",
    "retrieval_precision_issue",
    "generation_or_alignment_issue",
]

diagnostic_df["likely_issue"] = np.select(conditions, choices, default="mixed_or_minor")

diagnostic_summary = (
    diagnostic_df.groupby("likely_issue")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

low_score_cases = diagnostic_df.head(5)

display(diagnostic_summary)
display(low_score_cases)


,likely_issue,count
0,generation_or_alignment_issue,6
1,mixed_or_minor,2


,id,question,faithfulness,answer_relevancy,context_precision,context_recall,context_entity_recall,mean_score,likely_issue
0,FSR-Q04,報告中的第一類脆弱性是什麼？,0.75,0.0,0.000000,0.0,0.0,0.150000,generation_or_alignment_issue
1,FSR-Q05,第二類脆弱性在報告中如何定義？,0.80,0.0,0.000000,0.0,0.0,0.160000,generation_or_alignment_issue
2,FSR-Q06,第三類脆弱性與金融機構行為有何關聯？,1.00,0.0,0.866667,1.0,0.0,0.573333,generation_or_alignment_issue
3,FSR-Q02,報告如何描述金融穩定與聯準會雙重使命的關係？,1.00,0.0,1.000000,1.0,0.0,0.600000,generation_or_alignment_issue
4,FSR-Q07,第四類 funding risk 為何會導致 run 風險？,1.00,0.0,1.000000,1.0,0.0,0.600000,generation_or_alignment_issue


### Cell 7 結論：本次 RAGAS 基準評測結論（Vector RAG 實跑結果）

本次 `text-embedding-3-large + Top-5` 的 RAGAS 平均分數（8 題）：

| metric | mean |
|---|---:|
| faithfulness | 0.9438 |
| answer_relevancy | 0.3574 |
| context_precision | 0.7438 |
| context_recall | 0.7500 |
| context_entity_recall | 0.0250 |

重點解讀：

1. 與 TF-IDF 版本相比，`context_precision` 明顯提升，表示向量檢索確實改善了命中品質。
2. `answer_relevancy` 雖有改善，但仍偏低，代表回答與問題意圖對齊仍是短板。
3. `context_entity_recall` 仍非常低，表示關鍵實體層級的證據覆蓋不足，會造成答題細節偏差。

本輪診斷摘要：

- `generation_or_alignment_issue`: 6 題
- `mixed_or_minor`: 2 題

最低分題目：

- `FSR-Q04`（mean_score = 0.1500）
- `FSR-Q05`（mean_score = 0.1600）

可行動結論（下一輪優先順序）：

1. 先加強生成約束（evidence-only、引用檢查、證據不足回覆機制）。
2. 針對低分題做 query expansion 與題型化檢索規則。
3. 再重跑 RAGAS 驗證是否能拉升 `answer_relevancy` 與 `context_entity_recall`。

一句話總結：
`Vector RAG 已明顯改善檢索精準度，但主要瓶頸仍在答案與證據對齊，下一輪應優先做 evidence-grounded 生成控制。`
